# Badger_vision Dataset Analytics

Analyze your dataset before training — check for corrupt images, class
imbalance, bbox anomalies, and more.

Supports Badger Factory archives (`.7z`, `.zip`, etc.) and plain COCO / YOLO folders.

## 1 — Setup

In [ ]:
import subprocess
import sys

for pkg in ["tqdm", "py7zr", "rarfile"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
print(f"PyTorch: {torch.__version__}")

## 2 — Configure

Point to your dataset (folder or archive).

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE HERE                         ║
# ╚════════════════════════════════════════════════════════════╝

DATASET_PATH = "/path/to/your/dataset"  # folder or archive
TASK = "detection"    # "detection" | "keypoints" | "classification"

## 3 — Load & Prepare Dataset

In [ ]:
import importlib.util
import shutil
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "pyproject.toml").exists() else NOTEBOOK_DIR

spec = importlib.util.spec_from_file_location("linux_train", str(REPO_ROOT / "notebooks" / "linux_train.py"))
lt = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lt)

WORKSPACE = Path("badger_vision_workspace").resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)

dataset_path = Path(DATASET_PATH).resolve()
assert dataset_path.exists(), f"Dataset not found: {dataset_path}"

if dataset_path.is_file() and lt.is_archive(dataset_path):
    extract_dest = WORKSPACE / "dataset"
    if extract_dest.exists():
        shutil.rmtree(extract_dest)
    dataset_root = lt.extract_archive(dataset_path, extract_dest)
else:
    dataset_root = dataset_path

dataset_root = lt.resolve_dataset_root(dataset_root, TASK)
fmt = lt.detect_format(dataset_root)
print(f"Dataset root : {dataset_root}")
print(f"Format       : {fmt}")

if fmt == "badger_yolo":
    cls = dataset_root / "classes.txt"
    data_info = lt.prepare_badger_yolo(dataset_root, WORKSPACE, cls if cls.exists() else None)
elif fmt == "badger_classifier":
    data_info = lt.prepare_badger_classifier(dataset_root, WORKSPACE)
elif fmt == "coco_archive":
    data_info = lt.prepare_coco_archive(dataset_root, WORKSPACE)
elif fmt == "yolo_flat":
    data_info = lt.prepare_yolo_flat(dataset_root, WORKSPACE)
elif fmt == "coco_flat":
    data_info = lt.prepare_coco_flat(dataset_root, WORKSPACE)
else:
    raise RuntimeError(f"Unknown dataset format in {dataset_root}")

num_classes = lt.detect_num_classes(data_info["train_ann_file"])
num_train = lt.count_images(data_info["train_ann_file"])
print(f"Classes      : {num_classes}")
print(f"Train images : {num_train}")

## 4 — Dataset Statistics

In [ ]:
import json
from collections import Counter

with open(data_info["train_ann_file"]) as f:
    coco = json.load(f)

categories = {c["id"]: c["name"] for c in coco.get("categories", [])}
annotations = coco.get("annotations", [])
images = coco.get("images", [])

print(f"Total images      : {len(images)}")
print(f"Total annotations : {len(annotations)}")
print(f"Categories        : {len(categories)}")

# Class distribution
class_counts = Counter(categories.get(a["category_id"], "unknown") for a in annotations)
print(f"\nClass distribution:")
for name, count in class_counts.most_common():
    print(f"  {name:>30s} : {count:>6d}")

# Images with no annotations
annotated_ids = {a["image_id"] for a in annotations}
unannotated = [img for img in images if img["id"] not in annotated_ids]
if unannotated:
    print(f"\nWARNING: {len(unannotated)} images have no annotations")

## 5 — Bbox Analysis

In [ ]:
import numpy as np

if annotations and "bbox" in annotations[0]:
    bboxes = np.array([a["bbox"] for a in annotations])  # [x, y, w, h]
    widths = bboxes[:, 2]
    heights = bboxes[:, 3]
    areas = widths * heights
    aspects = widths / np.clip(heights, 1, None)

    print(f"Bbox widths  : min={widths.min():.0f}  max={widths.max():.0f}  mean={widths.mean():.1f}")
    print(f"Bbox heights : min={heights.min():.0f}  max={heights.max():.0f}  mean={heights.mean():.1f}")
    print(f"Bbox areas   : min={areas.min():.0f}  max={areas.max():.0f}  mean={areas.mean():.1f}")
    print(f"Aspect ratio : min={aspects.min():.2f}  max={aspects.max():.2f}  mean={aspects.mean():.2f}")

    # Tiny / huge bboxes
    tiny = np.sum(areas < 32 * 32)
    huge = np.sum(areas > 512 * 512)
    zero_dim = np.sum((widths <= 0) | (heights <= 0))
    if tiny:
        print(f"\nWARNING: {tiny} tiny bboxes (area < 32x32)")
    if huge:
        print(f"WARNING: {huge} large bboxes (area > 512x512)")
    if zero_dim:
        print(f"WARNING: {zero_dim} bboxes with zero width or height")
else:
    print("No bounding box annotations found.")

## 6 — Badger_vision Health Report

In [ ]:
import textwrap

# Write a data config so Badger_vision can analyze it
data_yaml = WORKSPACE / "data_analysis.yaml"
data_yaml.write_text(textwrap.dedent(f"""\
    img_dir: "{data_info['train_img_dir']}"
    ann_file: "{data_info['train_ann_file']}"
"""))

model_yaml = WORKSPACE / "model_analysis.yaml"
model_yaml.write_text(textwrap.dedent(f"""\
    model:
      name: BadgerResNeXt-Nano
      type: resnext
      backbone: resnext_nano
      neck_channels: 64
      head_depth: 2
      use_ghost: true
      num_classes: {num_classes}
      image_size: 640
    training:
      epochs: 1
"""))

from badger_vision.core.api import Badger_vision

model = Badger_vision(str(model_yaml))
report_path = model.analyze(data=str(data_yaml))

with open(report_path) as f:
    report = json.load(f)
for key, val in report.items():
    print(f"{key}: {val}")